<a href="https://colab.research.google.com/github/nullpoetry/3d-voxel-CA-bytebeat-midi-babylon/blob/main/Secure_Serverless_CLI_and_Handler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Secure Serverless CLI with Header Encryption
--------------------------------------------
Client: Encrypts a payload with a pre-shared key, sends via custom header.
Server: Decrypts the header, verifies, and processes the request.

Dependencies:
  pip install cryptography requests
"""

import os
import base64
import requests
from cryptography.fernet import Fernet

# --- CONFIGURATION (Shared Secret) ---
# In production, use environment variables or a secure Secret Manager.
KEY = Fernet.generate_key() # Replace with a persistent, secure key.
cipher_suite = Fernet(KEY)

# --- CLI CLIENT SIDE ---
def send_secure_request(payload: str, endpoint: str):
    """Encrypts payload and sends to serverless endpoint."""

    # 1. Encrypt the payload
    encrypted_payload = cipher_suite.encrypt(payload.encode())

    # 2. Prepare headers (Header Encryption Pattern)
    headers = {
        "X-Encrypted-Payload": encrypted_payload.decode(),
        "Content-Type": "application/json"
    }

    # 3. Send to API Gateway / Serverless Function URL
    response = requests.post(endpoint, headers=headers)
    return response.json()

# --- SERVERLESS FUNCTION SIDE (e.g., AWS Lambda) ---
def lambda_handler(event, context):
    """Decrypts and processes incoming request."""

    encrypted_header = event.get('headers', {}).get('x-encrypted-payload')

    if not encrypted_header:
        return {"statusCode": 403, "body": "Missing security header"}

    try:
        # 1. Decrypt the header
        decrypted_data = cipher_suite.decrypt(encrypted_header.encode())

        # 2. Process the business logic
        print(f"Decrypted request: {decrypted_data.decode()}")

        return {
            "statusCode": 200,
            "body": "Request Processed Securely"
        }

    except Exception as e:
        return {"statusCode": 401, "body": "Decryption failed: Unauthorized"}

# Usage Example:
# send_secure_request('{"command": "list_buckets"}', 'https://api.your-serverless-endpoint.com')